In [6]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re
from venn import venn
import json
import ast

from utils.selection import *

# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 
# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

In [ ]:
df = pd.read_csv(os.path.join(ANALYSIS_PATH,"dicom_files_df.csv"),sep=',')
print(df.info())
df.head()

- sort by unique sequence patient and study id

In [ ]:
unique_pairs = (df.sort_values(by="File_Name").drop_duplicates(subset=["Patient_ID", "Study_ID", "Sequence_ID"]).reset_index(drop=True))
unique_pairs.info()

In [9]:
unstructured={path:{} for path in unique_pairs['Full_Path']  }
output_path = os.path.join(ANALYSIS_PATH,"other_tags.json")   

#### reading of the dicom tags in separte file
---> in separate file  ./1_0_1_sequence_preselection_tags.py

In [11]:


with open(output_path, 'r', encoding='utf-8') as file:
    unstructured = json.load(file)

- TAG (0x0008, 0x0008) IMAGE_TYPE

In [ ]:
unique_IMAGE_attributes = []
for key, value in unstructured.items():
    val = value.get('Image_Type')
    if val is not None and val != 'None':
        if isinstance(val, str):
            try:
                val = ast.literal_eval(val)  
            except (SyntaxError, ValueError):
                print(f"Skipping invalid literal: {val}")
                continue  
        if isinstance(val, list):  # Check if val is a list
            unique_IMAGE_attributes.extend(val)
unique_IMAGE_attributes = list(set(unique_IMAGE_attributes))
# Create a DataFrame with the unique attributes
rows = []
for path, values in unstructured.items():
    attributes=values.get('Image_Type')
    row = {"Full_Path": path}
    for attr in unique_IMAGE_attributes:
        row[attr] = attr in attributes if attributes is not None else False
    rows.append(row)
df_Image_Type = pd.DataFrame(rows)
df_Image_Type.head()


- TAG (0x0008, 0x1030) STUDY_DESCRIPTION

In [ ]:
 #get all unique attibutes
unique_Study_Description_attributes = []
for key, value in unstructured.items():
    try:
        val=value["Study_Description"]
        if val is not None:
            unique_Study_Description_attributes.append(str(val))
    except:
        continue
unique_Study_Description_attributes=list(set(unique_Study_Description_attributes))
# Create a DataFrame with the unique attributes
rows = []
for path, values in unstructured.items():
    attributes=values.get('Study_Description')
    row = {"Full_Path": path}
    for attr in unique_Study_Description_attributes:
        row[attr] = attr in attributes if attributes is not None else False
    rows.append(row)
df_Study_Description_Type = pd.DataFrame(rows)
df_Study_Description_Type.head()

In [ ]:
merged_df = pd.merge(unique_pairs, df_Image_Type, on="Full_Path")
merged_df

- gem besprechung vom 10.1.25 sollten DWIS diese Image_type tag s haben : ['DERIVED', 'PRIMARY', 'DIFFUSION', 'NONE', 'TRACEW', 'ND', 'NORM']

# dwi

In [ ]:
variables_to_check = [  'DIFFUSION',  'TRACEW', 'ND', 'NORM'] #'NONE','DERIVED','PRIMARY',

filtered_df = merged_df[merged_df[variables_to_check].any(axis=1)]
print(filtered_df['Patient_ID'].nunique())


In [ ]:
variables_to_check = ['DIFFUSION','TRACEW','DERIVED', 'ND', 'PRIMARY']

filtered_df = merged_df[merged_df[variables_to_check].all(axis=1)]
print(filtered_df['Patient_ID'].nunique())
filtered_df['Sequence_ID_split'].unique()

In [ ]:
variables_to_check = ['DERIVED', 'PRIMARY', 'DIFFUSION', 'NONE', 'TRACEW', 'ND', 'NORM']

dwi_df = merged_df[merged_df[variables_to_check].all(axis=1)]



print(dwi_df['Patient_ID'].nunique())
dwi_df['Sequence_ID_split'].unique()



In [ ]:
dwi_df_merged=pd.merge(df,dwi_df[['Patient_ID','Study_ID','Sequence_ID']] , on=['Patient_ID','Study_ID','Sequence_ID'], how='inner')
dwi_df_merged.to_csv(os.path.join(ANALYSIS_PATH,"dwi_through_tags.csv"),index=False)
dwi_df_merged['Patient_ID'].nunique()

- the other tags described in the appendix of the study

dwi:
- repetoitn time msec/3000-4100/

echo time msec of 64-89
- flip angle  90-180
- imagning matrix 128x128,176x176 or 192x192
- in plane spacing 1.198-1.797
- section thickness 5-6.5 mm
- diffusion encoding gradients 0 500 and 1000 sec/mm2

corresponding DICOM


(0018,0080) Repetition Time                     DS: '3500'
(0018,0081) Echo Time                           DS: '89'
(0018,1314) Flip Angle                          DS: '90'

(0028,0030) Pixel Spacing                       DS: [1.796875, 1.796875]
(0018,0050) Slice Thickness                     DS: '5'

In [ ]:
Slice_Thickness


In [ ]:
 #get all unique attibutes
unique_Slice_Thickness_attributes = []
for key, value in unstructured.items():
    try:
        val=value["Slice_Thickness"]
        if val is not None:
            unique_Slice_Thickness_attributes.append(str(val))
    except:
        continue
unique_Slice_Thickness_attributes=list(set(unique_Slice_Thickness_attributes))
# Create a DataFrame with the unique attributes
rows = []
for path, values in unstructured.items():
    attributes=values.get('Slice_Thickness')
    row = {"Full_Path": path}
    for attr in unique_Slice_Thickness_attributes:
        row[attr] = attr in attributes if attributes is not None else False
    rows.append(row)
df_Slice_Thickness_Type = pd.DataFrame(rows)


# remove thos with none ande lower thatn 5 or more that 6.5
filtered_columns = [col for col in df_Slice_Thickness_Type.columns if col not in ['Full_Path', 'None']]
selection=[name for name in  filtered_columns if float(name) >=5 and float(name)<= 6.5]
selection.append('Full_Path')
dwi_on_slicethickness_df=df_Slice_Thickness_Type[selection]
dwi_slice_df=pd.merge(unique_pairs, dwi_on_slicethickness_df, on="Full_Path")

dwi_slice_df
dwi_slice_df2 = dwi_slice_df[dwi_slice_df[selection[:-1]].any(axis=1)]
dwi_slice_df2['Patient_ID'].nunique()

dwi_slice_df2

In [ ]:
 #get all unique attibutes
unique_Repetition_Time_attributes = []
for key, value in unstructured.items():
    try:
        val=value["Repetition_Time"]
        if val is not None:
            unique_Repetition_Time_attributes.append(str(val))
    except:
        continue
unique_Repetition_Time_attributes=list(set(unique_Repetition_Time_attributes))
# Create a DataFrame with the unique attributes
rows = []
for path, values in unstructured.items():
    attributes=values.get('Repetition_Time')
    row = {"Full_Path": path}
    for attr in unique_Repetition_Time_attributes:
        row[attr] = attr in attributes if attributes is not None else False
    rows.append(row)
df_Repetition_Time_Type = pd.DataFrame(rows)


# remove thos with none ande lower thatn 5 or more that 6.5
filtered_columns = [col for col in df_Repetition_Time_Type.columns if col not in ['Full_Path', 'None']]
selection=[name for name in  filtered_columns if float(name) >=3000 and float(name)<= 4100]
selection.append('Full_Path')
dwi_on_Repetition_Time_df=df_Repetition_Time_Type[selection]
dwi_Repetition_Time_df=pd.merge(unique_pairs, dwi_on_Repetition_Time_df, on="Full_Path")

dwi_Repetition_Time_df
dwi_Repetition_Time_df2 = dwi_Repetition_Time_df[dwi_Repetition_Time_df[selection[:-1]].any(axis=1)]
print(dwi_Repetition_Time_df2['Patient_ID'].nunique())

dwi_Repetition_Time_df2

repetiontion time

In [ ]:
 #get all unique attibutes
unique_Repetition_Time_attributes = []
for key, value in unstructured.items():
    try:
        val=value["Repetition_Time"]
        if val is not None:
            unique_Repetition_Time_attributes.append(str(val))
    except:
        continue
unique_Repetition_Time_attributes=list(set(unique_Repetition_Time_attributes))
# Create a DataFrame with the unique attributes
rows = []
for path, values in unstructured.items():
    attributes=values.get('Repetition_Time')
    row = {"Full_Path": path}
    for attr in unique_Repetition_Time_attributes:
        row[attr] = attr in attributes if attributes is not None else False
    rows.append(row)
df_Repetition_Time_Type = pd.DataFrame(rows)


# remove thos with none ande lower thatn 5 or more that 6.5
filtered_columns = [col for col in df_Repetition_Time_Type.columns if col not in ['Full_Path', 'None']]
selection=[name for name in  filtered_columns if float(name) >=3000 and float(name)<= 4100]
selection.append('Full_Path')
dwi_on_Repetition_Time_df=df_Repetition_Time_Type[selection]
dwi_Repetition_Time_df=pd.merge(unique_pairs, dwi_on_Repetition_Time_df, on="Full_Path")

dwi_Repetition_Time_df
dwi_Repetition_Time_df2 = dwi_Repetition_Time_df[dwi_Repetition_Time_df[selection[:-1]].any(axis=1)]
print(dwi_Repetition_Time_df2['Patient_ID'].nunique())

dwi_Repetition_Time_df2

echo time 64- 89

In [ ]:
 #get all unique attibutes
unique_Echo_Time_attributes = []
for key, value in unstructured.items():
    try:
        val=value["Echo_Time"]
        if val is not None:
            unique_Echo_Time_attributes.append(str(val))
    except:
        continue
unique_Echo_Time_attributes=list(set(unique_Echo_Time_attributes))
# Create a DataFrame with the unique attributes
rows = []
for path, values in unstructured.items():
    attributes=values.get('Echo_Time')
    row = {"Full_Path": path}
    for attr in unique_Echo_Time_attributes:
        row[attr] = attr in attributes if attributes is not None else False
    rows.append(row)
df_Echo_Time_Type = pd.DataFrame(rows)


# remove thos with none ande lower thatn 5 or more that 6.5
filtered_columns = [col for col in df_Echo_Time_Type.columns if col not in ['Full_Path', 'None']]
selection=[name for name in  filtered_columns if float(name) >=64 and float(name)<= 89]
selection.append('Full_Path')
dwi_on_Echo_Time_df=df_Echo_Time_Type[selection]
dwi_Echo_Time_df=pd.merge(unique_pairs, dwi_on_Echo_Time_df, on="Full_Path")

dwi_Echo_Time_df
dwi_Echo_Time_df2 = dwi_Echo_Time_df[dwi_Echo_Time_df[selection[:-1]].any(axis=1)]
print(dwi_Echo_Time_df2['Patient_ID'].nunique())

dwi_Echo_Time_df2

## t1

In [ ]:
variables_to_check =  ['ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM']

t1_df = merged_df[merged_df[variables_to_check].all(axis=1)]

t1_df['Patient_ID'].nunique()



In [ ]:
data = {
    'T1': set(t1_df['Patient_ID']),
    'DWI': set(dwi_df['Patient_ID'])
}
set_labels = {
    'T1': data['T1'],

    'DWI': data['DWI']
}
venn(set_labels)
plt.title("Venn Diagram for T1, DWI")
plt.show()

In [ ]:
dwi_df

In [ ]:
merged_dwi = pd.merge(df, dwi_df, on=['Patient_ID', 'Date', 'Sequence_ID'], how='right')
merged_t1 = pd.merge(df, t1_df, on=['Patient_ID', 'Date', 'Sequence_ID'], how='right')



merged_dwi.drop(columns=[ 'Full_Path_y', 'File_Name_y','Sequence_ID_split_y'], inplace=True)
merged_t1.drop(columns=[ 'Full_Path_y', 'File_Name_y','Sequence_ID_split_y'], inplace=True)

print(merged_dwi.info())
print(merged_t1.info())

In [ ]:
merged_dwi

In [ ]:
intersection_pat_ids_dwi_t1=set(set(merged_dwi['Patient_ID'])).intersection(set(merged_t1['Patient_ID']))

dict_t1_dwi={key:{} for key in intersection_pat_ids_dwi_t1}

for key in intersection_pat_ids_dwi_t1:
    dict_t1_dwi[key]['T1']=list(merged_t1[merged_t1['Patient_ID']==key]['Full_Path_x'])
    dict_t1_dwi[key]['DWI']=list(merged_dwi[merged_dwi['Patient_ID']==key]['Full_Path_x'])

data = []
for pat_id, dictionaries in dict_t1_dwi.items():
    n_t1=len(dictionaries['T1'])
    n_dwi=len(dictionaries['DWI']) 
    data.append({"Patient ID": pat_id, "T1 Paths Count": n_t1, "DWI Paths Count": n_dwi})
count_dwi_t1_first = pd.DataFrame(data)
count_dwi_t1_first

In [ ]:
dict_t1_dwi

In [ ]:
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_second_try'

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in dwi_t1_all_df['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]

print(target_nii_sequences[0])